# 开放数据集上的质检、覆盖和四级标注

还没有自有头戴设备时，用带手部位姿的开放数据把规格 v10 的三段跑通：**统一 episode → QC/产出率 → 覆盖空档**，并校验四级标注。差距分析见 `docs/gap_analysis.md`，定义见 `docs/headcam_data_spec.md` §8。

默认只跑**合成的 EgoDex 式 HDF5**（两条短片段），不下载数据。真实 EgoDex 测试集 `test.zip` 约 16.1GB，许可 CC BY-NC-ND，不要提交进仓库。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyxiaobai/robot/blob/main/notebooks/open_dataset_pipeline_colab.ipynb)


In [ ]:
%pip install -q numpy h5py


In [ ]:
import os
import subprocess
from pathlib import Path

root = Path.cwd()
if not (root / "scripts" / "demo_open_dataset_pipeline.py").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/skyxiaobai/robot.git", "/content/robot",
    ])
    os.chdir("/content/robot")
print("repo", Path.cwd())


## 合成样本

一条手腕在画面中央移动，一条置信度很低且手腕在画面外。产出率应约为 50%：只有通过片段级 QC 的帧计入可用数据。两条都缺 SUBTASK 和 INSTRUCTION，因为源数据只有整段描述。


In [ ]:
import subprocess
import sys
from pathlib import Path

out = Path("outputs/open_data_demo")
subprocess.check_call([
    sys.executable, "scripts/demo_open_dataset_pipeline.py",
    "--out", str(out),
])
print((out / "yield_episodes.csv").read_text(encoding="utf-8"))
print((out / "coverage_counts.csv").read_text(encoding="utf-8"))


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "scripts/validate_hierarchy.py",
    "examples/hierarchy_annotation.json", "--strict",
])


## 换成 EgoDex 测试集抽样

只在磁盘够的时候做。下载和解压都不在默认单元里。

```bash
curl -L "https://ml-site.cdn-apple.com/datasets/egodex/test.zip" -o test.zip
unzip test.zip
python scripts/demo_open_dataset_pipeline.py --input test --out outputs/egodex_sample --limit 20
```

`--limit 20` 只转换前 20 条 HDF5。转换结果里的世界系是 ARKit 原点，不是本仓库重新做的 SLAM。严格校验会失败，直到有人补上时间分段子任务和分手指令。训练仍走 `notebooks/act_pusht_colab.ipynb`（pusht）；统一 episode 还没有 LeRobot 导出，见 `docs/gap_analysis.md`。
